In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from fuzzywuzzy import process

/usr/local/lib/python3.11/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
ratings = pd.read_csv('/kaggle/input/movie-recommendation-system/ratings.csv')
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


In [3]:
movies  = pd.read_csv('/kaggle/input/movie-recommendation-system/movies.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
# Compute counts
movie_counts = ratings['movieId'].value_counts()
user_counts  = ratings['userId'].value_counts()

# Pick the top-N popular movies and active users
TOP_MOVIES = 35000
TOP_USERS  = 35000

top_movie_ids = movie_counts.nlargest(TOP_MOVIES).index
top_user_ids  = user_counts.nlargest(TOP_USERS).index

In [5]:
# Filter ratings down to that slice
core_ratings = ratings[
    ratings['movieId'].isin(top_movie_ids) &
    ratings['userId'].isin(top_user_ids)
]

core_ratings.head()

,userId,movieId,rating,timestamp
254,3,1,4.0,1439472215
255,3,29,4.5,1484754967
256,3,32,4.5,1439474635
257,3,50,5.0,1439474391
258,3,111,4.0,1484753849


In [6]:
movie_data = core_ratings.merge(movies, on='movieId')
movie_data.head()

,userId,movieId,rating,timestamp,title,genres
0,3,1,4.0,1439472215,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,3,29,4.5,1484754967,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
2,3,32,4.5,1439474635,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,3,50,5.0,1439474391,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
4,3,111,4.0,1484753849,Taxi Driver (1976),Crime|Drama|Thriller


In [7]:
user_item_matrix = core_ratings.pivot(index=['userId'], columns=['movieId'], values='rating').fillna(0)
user_item_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,208096,208104,208112,208295,208385,208715,208737,208747,208800,208939
userId,,,,,,,,,,,,,,,,,,,,,
3,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12,4.0,2.0,2.0,0.0,0.0,0.0,3.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19,0.0,3.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162521,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
162524,4.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
162529,2.0,4.0,1.0,0.0,2.0,2.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
movie_names = movies[['movieId', 'title']].drop_duplicates().set_index('movieId').reset_index()
movie_names

,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)
...,...,...
62418,209157,We (2018)
62419,209159,Window of the Soul (2001)
62420,209163,Bad Poems (2018)
62421,209169,A Girl Thing (2001)


In [9]:
# Define a KNN model on cosine similarity
cf_knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=10, n_jobs=-1)
cf_knn_model.fit(user_item_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=10)

In [10]:
def movie_recommender_engine(movie_name, matrix, cf_model, n_recs):
    
    # Extract movie index using fuzzy matching
    matched_title = process.extractOne(movie_name, movie_names['title'])
    movie_id = matched_title[2] if matched_title else None
    if movie_id is None:
        raise ValueError("Movie not found")

    # Compute neighbors
    distances, indices = cf_model.kneighbors(matrix.iloc[[movie_id]], n_neighbors=n_recs)
    movie_rec_ids = sorted(
        list(zip(indices.squeeze().tolist(), distances.squeeze().tolist())),
        key=lambda x: x[1]
    )[:0:-1]

    # Create recommendation DataFrame
    cf_recs = []
    for i in movie_rec_ids:
        cf_recs.append({'Title': movie_names['title'][i[0]], 'Distance': i[1]})
    df = pd.DataFrame(cf_recs, index=range(1, n_recs))
    return df

In [13]:
n_recs = 10
recommendations = movie_recommender_engine('Batman', user_item_matrix, cf_knn_model, n_recs)
print(recommendations)

                                     Title  Distance
1                         Leviathan (2014)  0.612352
2                Woman Buried Alive (1973)  0.610551
3                     Design Is One (2012)  0.610184
4  High and Low (Tengoku to jigoku) (1963)  0.609986
5              For Richer or Poorer (1997)  0.609188
6                    7 Days in Hell (2015)  0.604854
7                       Teen Spirit (2011)  0.602590
8                            Hamoun (1990)  0.589646
9              Ginger Ale Afternoon (1989)  0.579718
